# PDF 문서 로드 및 확인

In [14]:
from langchain_community.document_loaders import PyPDFLoader

FILE_PATH = "../data/가상Tech_업무가이드.pdf"

loader = PyPDFLoader(FILE_PATH)
docs = loader.load()

print(f"총 페이지 수: {len(docs)}")

총 페이지 수: 9


In [15]:
# 페이지별 글자 수와 앞부분 미리보기
for doc in docs:
    page = doc.metadata.get("page", 0) + 1
    print(f"[{page}페이지] 글자 수: {len(doc.page_content)}")
    print(doc.page_content[:100])
    print("-" * 80)

[1페이지] 글자 수: 252
VIRTUALTECH · INTERNAL HANDBOOK
가상Tech
업무 가이드
함께 일하는 방식부터 보안, 개발, 장애 대응, 복지까지
구성원이 매일 참고하는 실무 기준
 문서
--------------------------------------------------------------------------------
[2페이지] 글자 수: 851
가상Tech 업무 가이드
사내용 · v1.0
문서 책임: People Operations
02
CHAPTER 01
회사와 업무 원칙
빠르게 만들되, 고객 신뢰와 동료의 집중 시간을
--------------------------------------------------------------------------------
[3페이지] 글자 수: 869
가상Tech 업무 가이드
사내용 · v1.0
문서 책임: People Operations
03
CHAPTER 02
계정·기기·정보보안
계정 보호와 데이터 분류는 모든 구성원의 기본
--------------------------------------------------------------------------------
[4페이지] 글자 수: 935
가상Tech 업무 가이드
사내용 · v1.0
문서 책임: People Operations
04
CHAPTER 03
업무 도구와 커뮤니케이션
도구별 역할을 명확히 해 정보가 흩어지지
--------------------------------------------------------------------------------
[5페이지] 글자 수: 744
가상Tech 업무 가이드
사내용 · v1.0
문서 책임: People Operations
05
CHAPTER 04
문서 작성과 지식 관리
좋은 문서는 검색되고, 최신이며, 다음 행
--------------------------------------------------------------------------------
[6페이지

In [16]:
# 첫 페이지 메타데이터 확인
docs[0].metadata

{'producer': 'ReportLab PDF Library - (opensource)',
 'creator': '(unspecified)',
 'creationdate': '2026-09-26T11:41:50+09:00',
 'author': '가상Tech People Operations',
 'keywords': '',
 'moddate': '2026-09-26T11:41:50+09:00',
 'subject': 'RAG 실습용 가상 사내 업무 가이드',
 'title': '가상Tech 업무 가이드',
 'trapped': '/False',
 'source': '../data/가상Tech_업무가이드.pdf',
 'total_pages': 9,
 'page': 0,
 'page_label': '1'}

# 텍스트 분할 (Splitter)

- 문서 규모: 9페이지, 페이지당 약 250~950자 → 작은 사내 가이드 문서
- 선택: `RecursiveCharacterTextSplitter` (문단 → 줄 → 문장 → 단어 순으로 자연스러운 경계에서 분할)

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 분할 설정값
SPLIT_CONFIG = {
    "chunk_size": 500,     # 청크 하나의 최대 글자 수
    "chunk_overlap": 100,  # 앞뒤 청크가 겹치는 글자 수 (chunk_size의 약 20%)
    "separators": ["\n\n", "\n", ". ", "다. ", " ", ""],  # 한국어 문장 끝 포함
}

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=SPLIT_CONFIG["chunk_size"],
    chunk_overlap=SPLIT_CONFIG["chunk_overlap"],
    separators=SPLIT_CONFIG["separators"],
    length_function=len,
    add_start_index=True,  # 원문에서의 시작 위치를 metadata에 기록
)

split_docs = text_splitter.split_documents(docs)
print(f"원본 페이지 수: {len(docs)} → 청크 수: {len(split_docs)}")

원본 페이지 수: 9 → 청크 수: 19


In [18]:
# 청크 길이 분포와 샘플 확인
lengths = [len(d.page_content) for d in split_docs]
print(f"최소: {min(lengths)}, 최대: {max(lengths)}, 평균: {sum(lengths) / len(lengths):.0f}")

for i, d in enumerate(split_docs[:3]):
    print(f"[청크 {i}] page={d.metadata['page'] + 1}, start_index={d.metadata['start_index']}")
    print(d.page_content)
    print("-" * 80)

최소: 171, 최대: 498, 평균: 419
[청크 0] page=1, start_index=0
VIRTUALTECH · INTERNAL HANDBOOK
가상Tech
업무 가이드
함께 일하는 방식부터 보안, 개발, 장애 대응, 복지까지
구성원이 매일 참고하는 실무 기준
 문서 버전
 시행일
 문서 등급
 문의
v1.0
2026-09-01
사내 공개
People Operations
이 문서의 목적
가상Tech 구성원이 반복해서 찾는 사내 정책과 표준 절차를 한 문서에 정리했습니다.
실제 회사가 아닌 RAG 검색·질의응답 실습용 가상 문서입니다.
--------------------------------------------------------------------------------
[청크 1] page=2, start_index=0
가상Tech 업무 가이드
사내용 · v1.0
문서 책임: People Operations
02
CHAPTER 01
회사와 업무 원칙
빠르게 만들되, 고객 신뢰와 동료의 집중 시간을 지킵니다.
 
1.1 회사 개요
가상Tech는 기업용 협업 자동화 SaaS인 ‘FlowPilot’을 개발·운영하는 가상 기업입니다. 본사는 서울 성수동에 있으며
제품개발, 고객성공, 영업, 경영지원 조직으로 구성됩니다. 표준 근무시간은 평일 10:00-19:00이고, 핵심 협업시간은
11:00-16:00입니다.
1.2 우리가 일하는 다섯 가지 원칙
 • 고객 영향부터 판단한다: 기능보다 고객 데이터와 서비스 연속성을 우선한다.
 • 기록이 회의를 이긴다: 결정, 근거, 담당자, 기한을 문서에 남긴다.
 • 작게 배포하고 빠르게 검증한다: 변경 범위를 줄이고 되돌릴 수 있게 만든다.
 • 문제는 사람보다 시스템에서 찾는다: 회고는 비난 없이 재발 방지에 집중한다.
--------------------------------------------------------------------------------
[청크 2] page=2, start_inde

# 임베딩 (Embedding)

- 모델: OpenAI `text-embedding-3-small` (한국어 지원, 저렴함, `dimensions`로 차원 축소 가능)
- LangChain 1.x 기준 `langchain_openai`만 사용 (레거시 `langchain_classic`의 `CacheBackedEmbeddings`는 쓰지 않음)
- 임베딩 재계산 방지는 다음 단계에서 벡터스토어를 디스크에 저장(persist)하는 방식으로 처리

In [19]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

# 임베딩 설정값
EMBED_CONFIG = {
    "model": "text-embedding-3-small",  # 더 높은 정확도가 필요하면 "text-embedding-3-large"
    "dimensions": 1536,                  # 3-small 기본 차원 (줄이면 저장 공간/검색 속도 이득)
    "chunk_size": 100,                   # 한 번의 API 요청에 묶어 보낼 청크 수
}

embeddings = OpenAIEmbeddings(
    model=EMBED_CONFIG["model"],
    dimensions=EMBED_CONFIG["dimensions"],
    chunk_size=EMBED_CONFIG["chunk_size"],
)

In [20]:
# 청크 임베딩 결과 확인
doc_vectors = embeddings.embed_documents([d.page_content for d in split_docs])
print(f"임베딩한 청크 수: {len(doc_vectors)}, 벡터 차원: {len(doc_vectors[0])}")

query_vector = embeddings.embed_query("연차 휴가는 어떻게 신청하나요?")
print(f"질문 벡터 차원: {len(query_vector)}")

임베딩한 청크 수: 19, 벡터 차원: 1536
질문 벡터 차원: 1536


# 벡터 DB (Chroma)

## Chroma vs FAISS

| 항목 | Chroma | FAISS |
|---|---|---|
| 정체 | 벡터 **데이터베이스** | 벡터 **검색 라이브러리** (Meta) |
| 저장 | `persist_directory` 지정 시 자동으로 디스크 저장 | `save_local()` / `load_local()`로 직접 저장·로드 |
| 메타데이터 필터 | `filter={"page": 3}` 같은 조건 검색 기본 지원 | LangChain 래퍼에서 가능하지만 검색 후 거르는 방식이라 제한적 |
| 추가·수정·삭제 | id 기반 `add` / `update` / `delete` 자연스럽게 지원 | 추가는 쉽지만 수정·삭제는 불편함 |
| 속도·규모 | 수십만 건 이하에서 충분히 빠름 | 수백만~수억 건, GPU 가속 등 대규모에 강함 |
| 서버 모드 | 로컬 파일 / 클라이언트-서버 둘 다 가능 | 서버 없음 (메모리 인덱스) |

## Chroma를 고른 이유

- 사내 가이드처럼 **문서가 자주 수정**되는 경우 id 기반 갱신이 편함
- 페이지·챕터 같은 **메타데이터 필터 검색**을 바로 쓸 수 있음
- 디스크에 저장해 두면 **다시 실행할 때 임베딩 API를 호출하지 않음** (임베딩 캐시 역할)

In [ ]:
import hashlib

from langchain_chroma import Chroma

# 벡터 DB 설정값
VECTORSTORE_CONFIG = {
    "collection_name": "virtualtech_guide",
    "persist_directory": "./chroma_db",  # 디스크 저장 위치
    "distance": "cosine",                # 유사도 계산 방식 (cosine / l2 / ip)
}

vectorstore = Chroma(
    collection_name=VECTORSTORE_CONFIG["collection_name"],
    embedding_function=embeddings,
    persist_directory=VECTORSTORE_CONFIG["persist_directory"],
    collection_metadata={"hnsw:space": VECTORSTORE_CONFIG["distance"]},
)

# 청크 내용 + 위치로 고정 id 생성 → 재실행해도 중복 저장되지 않음
ids = [
    hashlib.sha256(f'{d.metadata["page"]}-{d.metadata["start_index"]}-{d.page_content}'.encode()).hexdigest()
    for d in split_docs
]

# 이미 저장된 청크는 건너뛰고 새 청크만 임베딩해서 추가
existing_ids = set(vectorstore.get(ids=ids)["ids"])
new_docs = [d for d, i in zip(split_docs, ids) if i not in existing_ids]
new_ids = [i for i in ids if i not in existing_ids]

if new_docs:
    vectorstore.add_documents(documents=new_docs, ids=new_ids)

print(f"새로 추가한 청크: {len(new_docs)}, 컬렉션 전체 청크: {vectorstore._collection.count()}")

In [ ]:
# 유사도 검색 테스트 (점수: cosine 거리, 낮을수록 유사)
query = "연차 휴가는 어떻게 신청하나요?"
results = vectorstore.similarity_search_with_score(query, k=3)

for doc, score in results:
    print(f"[page={doc.metadata['page'] + 1}, score={score:.4f}]")
    print(doc.page_content[:200])
    print("-" * 80)

# 메타데이터 필터 검색 예시: 8페이지(휴가·비용·복지)에서만 검색
filtered = vectorstore.similarity_search(query, k=2, filter={"page": 7})
print(f"필터 검색 결과 페이지: {[d.metadata['page'] + 1 for d in filtered]}")

# 리트리버 (Retriever)

## 후보 비교

| 방식 | 동작 | 장점 | 단점 |
|---|---|---|---|
| similarity | 질문과 가장 가까운 k개 | 단순, 빠름 | 비슷한 내용의 청크가 중복으로 뽑힘 |
| **MMR** (선택) | 관련성 + 서로 다른 내용을 함께 고려 | overlap으로 겹친 청크 중복 제거, 다양한 근거 확보 | 후보(fetch_k)를 더 가져오므로 약간 느림 |
| similarity_score_threshold | 점수 기준 이상만 반환 | 관련 없는 청크 차단 | 기준값 튜닝 필요, 결과 0개 가능 |
| Hybrid (BM25 + 벡터) | 키워드 검색과 의미 검색 결합 | 고유명사·숫자 검색에 강함 | `EnsembleRetriever`가 레거시 `langchain_classic`에 있음 |
| Multi-Query / Re-ranking | LLM으로 질문 확장, 또는 재정렬 모델 사용 | 정확도 향상 | LLM/추가 모델 호출 비용 |

## MMR을 고른 이유

- `chunk_overlap=100`으로 인접 청크 내용이 겹쳐서, similarity만 쓰면 거의 같은 청크가 여러 개 뽑힘
- 문서가 작아 Hybrid·Re-ranking까지는 과함 → 먼저 MMR로 시작하고 부족하면 확장

In [ ]:
# 리트리버 설정값
RETRIEVER_CONFIG = {
    "search_type": "mmr",
    "k": 4,              # 최종으로 LLM에 넘길 청크 수
    "fetch_k": 12,       # MMR이 다양성을 고르기 전에 먼저 가져올 후보 수
    "lambda_mult": 0.7,  # 1에 가까울수록 관련성, 0에 가까울수록 다양성 우선
}

retriever = vectorstore.as_retriever(
    search_type=RETRIEVER_CONFIG["search_type"],
    search_kwargs={
        "k": RETRIEVER_CONFIG["k"],
        "fetch_k": RETRIEVER_CONFIG["fetch_k"],
        "lambda_mult": RETRIEVER_CONFIG["lambda_mult"],
    },
)

# similarity와 MMR 결과 비교
query = "연차 휴가는 어떻게 신청하나요?"
similarity_docs = vectorstore.as_retriever(search_kwargs={"k": RETRIEVER_CONFIG["k"]}).invoke(query)
mmr_docs = retriever.invoke(query)

print("similarity:", [(d.metadata["page"] + 1, d.metadata["start_index"]) for d in similarity_docs])
print("mmr       :", [(d.metadata["page"] + 1, d.metadata["start_index"]) for d in mmr_docs])

# RAG 체인 (LLM 연결)

- LCEL(`|` 파이프)로 직접 구성 → 레거시 `create_retrieval_chain` 등 `langchain_classic` 사용 안 함
- 모델: `gpt-5-mini`
- 답변 끝에 근거 페이지를 표시하고, 문서에 없는 내용은 모른다고 답하도록 프롬프트 작성

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

# LLM 설정값
LLM_CONFIG = {
    "model": "gpt-5-mini",
}

llm = ChatOpenAI(model=LLM_CONFIG["model"])

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 가상Tech 사내 업무 가이드 도우미야.\n"
     "아래 [문서]에 있는 내용만 근거로 한국어로 답해.\n"
     "답을 찾았으면 마지막 줄에 근거 페이지를 '(출처: N페이지)' 형식으로 적어.\n"
     "문서에 없는 내용이면 출처 없이 '가이드에서 찾을 수 없습니다'라고만 답해.\n\n"
     "[문서]\n{context}"),
    ("human", "{question}"),
])


def format_docs(docs):
    # 청크마다 페이지 번호를 붙여서 LLM이 출처를 적을 수 있게 함
    return "\n\n".join(f"[{d.metadata['page'] + 1}페이지]\n{d.page_content}" for d in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [36]:
# 질문 테스트 (마지막 질문은 문서에 없는 내용)
final_questions = [
"연차를 사용하려면 최소 며칠 전에 신청해야 하나요?",
"서울 출장 시 숙박비는 최대 얼마까지 지원되나요?",
"출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?",
"회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?",
"재택근무를 신청하려면 어떤 조건을 충족해야 하나요?",
"직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?",
"출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?",
"개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?",
"업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 무엇인가요?",
"제주도로 2박 3일 출장을 가는 경우 숙박비 지원 한도와 출장비 정산 기한을 각각 알려주세요.",
]

for i, q in enumerate(final_questions):
    print(f"Q-{i+1}. {q}")
    print(f"A-{i+1}. {rag_chain.invoke(q)}")
    print("=" * 80)

Q-1. 연차를 사용하려면 최소 며칠 전에 신청해야 하나요?
A-1. 가이드에서 찾을 수 없습니다
Q-2. 서울 출장 시 숙박비는 최대 얼마까지 지원되나요?
A-2. 서울 출장 시 숙박비는 1박 15만원(세금 포함)까지 지원됩니다. (출처: 8페이지)
Q-3. 출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?
A-3. 가이드에서 찾을 수 없습니다
Q-4. 회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?
A-4. 가이드에서 찾을 수 없습니다
Q-5. 재택근무를 신청하려면 어떤 조건을 충족해야 하나요?
A-5. 간단히 정리하면 다음 조건을 충족해야 합니다.

- 재택근무는 주 2회 기준(주 단위)입니다. 이월되지 않습니다.  
- 매주 월요일까지 캘린더에 ‘WFH’로 표시해야 합니다.  
- 고객 방문이나 건강상 사유 등 예외는 팀장과 People Ops의 승인이 필요합니다.  
- 재택이라도 근무 규정은 동일하게 적용됩니다(유연근무 08:00–11:00 사이 출근, 일 8시간 근무).  
- 응답 기대는 업무시간 기준 4시간입니다.

(출처: 2페이지, 9페이지)
Q-6. 직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?
A-6. 연 120만원까지 지원합니다. 교육·도서·컨퍼런스 비용에 해당하며, 지원 전 LearnHub에서 사전 승인을 받아야 합니다.  
(출처: 8페이지)
Q-7. 출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?
A-7. 출장 중 택시비는 업무상 필요할 때 실비로 지원됩니다. 예시로는 23:00 이후 퇴근, 대중교통 미운행, 고객 일정 등 업무상 필요가 있을 때 해당됩니다. (출처: 8페이지)
Q-8. 개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?
A-8. 가이드에서 찾을 수 없습니다
Q-9. 업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 무엇인가요?
A-9. 감지자는 #help-urgent에 ‘증상, 시작 시각, 영향, 관측 링크’

# 하이브리드 리트리버 (BM25 + 벡터, Ensemble)

- **BM25**: 키워드 일치 검색 → `LearnHub`, `#help-urgent`, `15만원` 같은 고유명사·숫자에 강함
- **벡터(MMR)**: 의미 검색 → "쉬고 싶을 때" 같은 표현이 달라도 찾음
- 두 결과를 **RRF(Reciprocal Rank Fusion)** 로 합침: 각 결과의 순위로 `weight / (rrf_k + 순위)` 점수를 더해 최종 순위 결정
- 레거시 `EnsembleRetriever` 대신 `langchain_core`의 `BaseRetriever`를 상속해 직접 구현
- 한국어는 띄어쓰기만으로 나누면 "휴가는", "휴가를"이 다른 단어가 되므로 형태소 분석기 **Kiwi**로 토큰화

In [30]:
from kiwipiepy import Kiwi
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from rank_bm25 import BM25Okapi

# 하이브리드 설정값
HYBRID_CONFIG = {
    "k": 4,               # 최종으로 LLM에 넘길 청크 수
    "candidate_k": 5,     # BM25·벡터 각각에서 가져올 후보 수
                          # (전체 19청크 중 8개씩 가져오면 두 결과가 거의 겹쳐서 BM25 1위 청크가 밀려남 → 5로 실험 후 결정)
    "bm25_weight": 0.5,   # 키워드 검색 가중치
    "vector_weight": 0.5, # 의미 검색 가중치
    "rrf_k": 60,          # RRF 상수 (클수록 순위 차이의 영향이 완만해짐)
}

kiwi = Kiwi()


def tokenize(text: str) -> list[str]:
    # 명사·동사·영어·숫자 등 의미 있는 형태소만 남김 (조사·어미 제외)
    return [t.form.lower() for t in kiwi.tokenize(text) if t.tag[0] in ("N", "V", "S", "X") and t.tag != "SF"]


class HybridRetriever(BaseRetriever):
    """BM25(키워드) + 벡터 리트리버 결과를 RRF로 합치는 리트리버

    BaseRetriever를 상속하고 _get_relevant_documents만 구현하면
    invoke / batch / ainvoke, LCEL 파이프(|), 콜백·트레이싱을 그대로 쓸 수 있음
    """

    documents: list[Document]
    vector_retriever: BaseRetriever
    bm25: BM25Okapi
    config: dict

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> list[Document]:
        cfg = self.config

        # 1) BM25 키워드 검색
        scores = self.bm25.get_scores(tokenize(query))
        bm25_top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[: cfg["candidate_k"]]
        bm25_docs = [self.documents[i] for i in bm25_top if scores[i] > 0]

        # 2) 벡터 검색 (run_manager.get_child()로 넘겨서 트레이싱에서 하이브리드 검색의 하위 단계로 기록)
        vector_docs = self.vector_retriever.invoke(query, config={"callbacks": run_manager.get_child()})

        # 3) RRF로 두 결과 합치기 (같은 청크는 페이지+시작위치로 판별)
        fused, doc_by_key = {}, {}
        for docs_, weight in ((bm25_docs, cfg["bm25_weight"]), (vector_docs, cfg["vector_weight"])):
            for rank, d in enumerate(docs_, start=1):
                key = (d.metadata["page"], d.metadata["start_index"])
                fused[key] = fused.get(key, 0) + weight / (cfg["rrf_k"] + rank)
                doc_by_key[key] = d

        ranked = sorted(fused, key=fused.get, reverse=True)[: cfg["k"]]
        return [doc_by_key[key] for key in ranked]


hybrid_retriever = HybridRetriever(
    documents=split_docs,
    vector_retriever=vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": HYBRID_CONFIG["candidate_k"],
            "fetch_k": HYBRID_CONFIG["candidate_k"] * 3,
            "lambda_mult": RETRIEVER_CONFIG["lambda_mult"],
        },
    ),
    bm25=BM25Okapi([tokenize(d.page_content) for d in split_docs]),
    config=HYBRID_CONFIG,
)

In [31]:
# MMR vs 하이브리드 비교
# (질문, 정답 청크에 반드시 들어 있어야 하는 단어) → 1순위 청크에 그 단어가 있는지 확인
compare_questions = [
    ("LearnHub에서는 뭘 신청하나요?", "LearnHub"),              # 키워드형
    ("#help-urgent 채널은 언제 쓰나요?", "#help-urgent"),       # 키워드형
    ("출장 숙박비 한도가 얼마예요?", "15만원"),                   # 숫자형
    ("몸이 아파서 쉬어야 하면 어떻게 해요?", "병가"),              # 의미형 (다른 표현)
    ("서비스가 갑자기 멈추면 누가 지휘하나요?", "장애"),           # 의미형
]

for q, keyword in compare_questions:
    print(f"Q. {q}  (정답 키워드: {keyword})")
    for name, r in (("MMR   ", retriever), ("Hybrid", hybrid_retriever)):
        found = r.invoke(q)
        pages = [d.metadata["page"] + 1 for d in found]
        top1_hit = "O" if keyword in found[0].page_content else "X"
        any_hit = sum(keyword in d.page_content for d in found)
        print(f"  {name} 페이지={pages}  1순위 적중={top1_hit}  4개 중 포함={any_hit}")
    print("-" * 80)

Q. LearnHub에서는 뭘 신청하나요?  (정답 키워드: LearnHub)
  MMR    페이지=[8, 9, 3, 4]  1순위 적중=X  4개 중 포함=0
  Hybrid 페이지=[8, 3, 8, 2]  1순위 적중=X  4개 중 포함=1
--------------------------------------------------------------------------------
Q. #help-urgent 채널은 언제 쓰나요?  (정답 키워드: #help-urgent)
  MMR    페이지=[4, 9, 9, 5]  1순위 적중=O  4개 중 포함=3
  Hybrid 페이지=[4, 5, 9, 5]  1순위 적중=O  4개 중 포함=2
--------------------------------------------------------------------------------
Q. 출장 숙박비 한도가 얼마예요?  (정답 키워드: 15만원)
  MMR    페이지=[8, 8, 9, 4]  1순위 적중=O  4개 중 포함=2
  Hybrid 페이지=[8, 8, 3, 9]  1순위 적중=O  4개 중 포함=2
--------------------------------------------------------------------------------
Q. 몸이 아파서 쉬어야 하면 어떻게 해요?  (정답 키워드: 병가)
  MMR    페이지=[3, 7, 5, 1]  1순위 적중=X  4개 중 포함=0
  Hybrid 페이지=[9, 3, 9, 7]  1순위 적중=X  4개 중 포함=0
--------------------------------------------------------------------------------
Q. 서비스가 갑자기 멈추면 누가 지휘하나요?  (정답 키워드: 장애)
  MMR    페이지=[3, 5, 8, 4]  1순위 적중=X  4개 중 포함=1
  Hybrid 페이지=[9, 5, 7, 3]  1순위 적중=O  4개 중 

## 비교 결과 정리 (실험값, `O/X`=1순위 적중, 숫자=4개 중 정답 키워드 포함 청크 수)

| 질문 | MMR | Hybrid (candidate_k=5) |
|---|---|---|
| LearnHub (키워드) | X0 | X1 ← BM25 덕분에 정답 청크 포함 |
| #help-urgent (키워드) | O3 | O2 |
| 숙박 한도 15만원 (숫자) | O2 | O2 |
| 몸이 아파서 쉬려면 (의미 → 병가) | X0 | X0 ← 둘 다 실패 |
| 서비스 멈춤 지휘 (의미 → 장애) | X1 | O3 ← 1순위 개선 |

- 하이브리드가 **1순위 적중 3/5 (MMR 2/5)**, 정답 청크를 놓친 질문도 1개 줄어듦 → 이 문서에는 하이브리드가 적절
- `candidate_k`를 8로 두면 BM25·벡터 후보가 거의 겹쳐서 **양쪽 모두에 나온 청크만 상위권**에 오고, BM25 단독 1위(LearnHub)가 밀려남
- "아파서 → 병가"처럼 **단어가 전혀 다른 질문**은 두 방식 모두 실패 → LLM으로 질문을 바꿔 쓰는 Query Rewriting이 다음 개선 포인트
- 질문 5개짜리 소규모 실험이라, 제대로 결정하려면 평가 세트를 늘려야 함

# Query Rewriting (질문 재작성)

- 문제: "몸이 아파서 쉬어야 하면?"처럼 **사용자 표현과 문서 용어가 다르면** BM25·벡터 둘 다 못 찾음 (문서에는 "병가")
- 해결: 검색 전에 LLM이 질문을 **사내 문서에 나올 법한 용어로 바꾼 검색어**로 재작성
  - 예) "몸이 아파서 쉬어야 하면?" → "병가 신청 절차 증빙"
- 검색에는 재작성된 질문을, **답변 생성에는 원래 질문**을 사용 (사용자 의도 유지)
- 비용: 질문당 LLM 호출 1회 추가

In [32]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 사내 업무 가이드 검색어 생성기야.\n"
     "가이드 목차: 회사와 업무 원칙, 근무 방식, 계정·기기·정보보안, 업무 도구와 커뮤니케이션, "
     "문서 작성과 지식 관리, 개발·변경·배포, 장애 대응과 고객 커뮤니케이션, 휴가·비용·복지, FAQ와 연락처\n"
     "사용자 질문을 가이드 문서에 실제로 쓰였을 법한 공식 용어로 바꾼 검색어 한 줄로 만들어.\n"
     "- 일상 표현은 사내 용어로 바꿔 (예: 쓴 돈 돌려받기 → 비용 정산, 노트북 잃어버림 → 기기 분실 신고)\n"
     "- 질문에 있는 고유명사·숫자·채널명은 그대로 유지해\n"
     "- 설명 없이 검색어만 출력해"),
    ("human", "{question}"),
])

rewrite_chain = rewrite_prompt | llm | StrOutputParser()

# 재작성 결과 확인
for q, _ in compare_questions:
    print(f"{q}  →  {rewrite_chain.invoke({'question': q})}")

LearnHub에서는 뭘 신청하나요?  →  LearnHub 교육 과정 신청
#help-urgent 채널은 언제 쓰나요?  →  #help-urgent 채널 사용 기준 및 응답 절차
출장 숙박비 한도가 얼마예요?  →  출장비 지급기준(숙박비)
몸이 아파서 쉬어야 하면 어떻게 해요?  →  병가 신청 및 결근 신고 절차
서비스가 갑자기 멈추면 누가 지휘하나요?  →  서비스 장애 발생 시 사고 지휘체계 및 책임자


In [33]:
# 재작성 → 하이브리드 검색 리트리버 (LCEL로 연결하면 이것도 하나의 Runnable)
rewrite_retriever = (lambda q: {"question": q}) | rewrite_chain | hybrid_retriever

# MMR vs Hybrid vs Rewrite+Hybrid 비교
for q, keyword in compare_questions:
    print(f"Q. {q}  (정답 키워드: {keyword})")
    for name, r in (("MMR           ", retriever), ("Hybrid        ", hybrid_retriever), ("Rewrite+Hybrid", rewrite_retriever)):
        found = r.invoke(q)
        pages = [d.metadata["page"] + 1 for d in found]
        top1_hit = "O" if keyword in found[0].page_content else "X"
        any_hit = sum(keyword in d.page_content for d in found)
        print(f"  {name} 페이지={pages}  1순위 적중={top1_hit}  4개 중 포함={any_hit}")
    print("-" * 80)

Q. LearnHub에서는 뭘 신청하나요?  (정답 키워드: LearnHub)
  MMR            페이지=[8, 9, 3, 4]  1순위 적중=X  4개 중 포함=0
  Hybrid         페이지=[8, 3, 8, 2]  1순위 적중=X  4개 중 포함=1
  Rewrite+Hybrid 페이지=[8, 2, 9, 8]  1순위 적중=O  4개 중 포함=1
--------------------------------------------------------------------------------
Q. #help-urgent 채널은 언제 쓰나요?  (정답 키워드: #help-urgent)
  MMR            페이지=[4, 9, 9, 5]  1순위 적중=O  4개 중 포함=3
  Hybrid         페이지=[4, 5, 9, 5]  1순위 적중=O  4개 중 포함=2
  Rewrite+Hybrid 페이지=[4, 8, 2, 4]  1순위 적중=O  4개 중 포함=2
--------------------------------------------------------------------------------
Q. 출장 숙박비 한도가 얼마예요?  (정답 키워드: 15만원)
  MMR            페이지=[8, 8, 9, 4]  1순위 적중=O  4개 중 포함=2
  Hybrid         페이지=[8, 8, 3, 9]  1순위 적중=O  4개 중 포함=2
  Rewrite+Hybrid 페이지=[8, 8, 9, 7]  1순위 적중=O  4개 중 포함=2
--------------------------------------------------------------------------------
Q. 몸이 아파서 쉬어야 하면 어떻게 해요?  (정답 키워드: 병가)
  MMR            페이지=[3, 7, 5, 1]  1순위 적중=X  4개 중 포함=0
  Hybrid         페이지=[9, 3, 9, 7]  1

## 비교 결과 정리 (실험값)

| 질문 | 재작성된 검색어 | MMR | Hybrid | Rewrite+Hybrid |
|---|---|---|---|---|
| LearnHub | LearnHub 교육과정 신청 | X0 | X1 | **O1** |
| #help-urgent | #help-urgent 채널 사용 기준 | O3 | O2 | X3 |
| 숙박 한도 15만원 | 출장비(숙박) 지급 한도 | O2 | O2 | O2 |
| 아파서 쉬려면 → 병가 | 병가 신청 절차 | X0 | X0 | **X1** ← 처음으로 정답 청크 포함 |
| 서비스 멈춤 → 장애 | 서비스 장애 발생 시 지휘 체계 | X1 | O3 | O2 |

- 정답 청크를 **하나도 못 가져온 질문**: MMR 2개 → Hybrid 1개 → **Rewrite+Hybrid 0개**
- 1순위 적중은 Hybrid·Rewrite+Hybrid 모두 3/5 (#help-urgent는 재작성 후 1순위가 다른 청크로 바뀜)
- LLM은 1순위만이 아니라 4개 청크를 모두 읽으므로, "정답 청크를 놓치지 않는 것"이 더 중요 → 최종 체인은 Rewrite+Hybrid 사용
- 재작성 결과는 LLM이 만들기 때문에 실행마다 조금씩 달라질 수 있음

# 최종 RAG 체인 (Query Rewriting + Hybrid)

질문 → **재작성된 검색어**로 하이브리드 검색 → **원래 질문** + 검색 결과로 답변 생성

In [35]:
final_rag_chain = (
    {"context": rewrite_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

final_questions = [
"연차를 사용하려면 최소 며칠 전에 신청해야 하나요?",
"서울 출장 시 숙박비는 최대 얼마까지 지원되나요?",
"출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?",
"회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?",
"재택근무를 신청하려면 어떤 조건을 충족해야 하나요?",
"직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?",
"출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?",
"개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?",
"업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 무엇인가요?",
"제주도로 2박 3일 출장을 가는 경우 숙박비 지원 한도와 출장비 정산 기한을 각각 알려주세요.",
]

for i, q in enumerate(final_questions):
    print(f"Q-{i+1}. {q}")
    print(f"A-{i+1}. {final_rag_chain.invoke(q)}")
    print("=" * 80)

Q-1. 연차를 사용하려면 최소 며칠 전에 신청해야 하나요?
A-1. 연차는 최소 1일 전 신청해야 합니다. 단, 3일 이상 연차는 2주 전 팀장과 협의해야 합니다. (출처: 8페이지)
Q-2. 서울 출장 시 숙박비는 최대 얼마까지 지원되나요?
A-2. 숙박비는 1박당 최대 15만원(세금 포함)까지 지원됩니다.
(출처: 8페이지)
Q-3. 출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?
A-3. 가이드에서 찾을 수 없습니다
Q-4. 회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?
A-4. 가이드에서 찾을 수 없습니다
Q-5. 재택근무를 신청하려면 어떤 조건을 충족해야 하나요?
A-5. 재택근무는 주 2회 기준으로 주 단위 적용되며 이월할 수 없습니다. 다만 고객 방문이나 건강상 사유 등의 예외는 팀장과 People Ops의 승인이 필요합니다. (출처: 9페이지)
Q-6. 직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?
A-6. 연간 120만원 한도로 교육·도서·컨퍼런스 비용을 지원합니다. 이용 전 LearnHub에서 사전 승인을 받아야 합니다.
(출처: 8페이지)
Q-7. 출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?
A-7. 출장 중 택시비는 23:00 이후 퇴근할 때, 대중교통이 운행하지 않을 때, 또는 고객 일정 등 업무상 필요가 있을 때 실비로 지원됩니다. 
(출처: 8페이지)
Q-8. 개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?
A-8. 개인 사정으로 출근이 늦어질 경우, 반차로 처리하려면 당일 10:00 전 팀장 승인을 받아야 합니다. (출처: 8페이지)
Q-9. 업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 무엇인가요?
A-9. 감지자는 즉시 #help-urgent 채널에 ‘증상, 시작 시각, 영향, 관측 링크’를 게시하고 PagerDuty를 호출해야 합니다. (출처: 7페이지)
Q-10. 제주도로 2박 3일 출장을 가는 경우 숙박